In [4]:
import pandas as pd
import numpy as np
from udrud_framework import detect_plateau_and_estimate
from joblib import Parallel, delayed
import multiprocessing

In [5]:
def compute_single_scenario(income, weight, k_range, window_size):
    """Function to run one sensitivity scenario."""
    # We call your framework's driver
    gamma_hat, best_k = detect_plateau_and_estimate(income, weight, k_range, window_size)
    return {'w': window_size, 'gamma_hat': gamma_hat, 'k^*': best_k}

# Number of CPU cores to use
n_cores = multiprocessing.cpu_count()

In [6]:
# Setup parameters
k_min = 20
p = 0.05  # tail depth (5% of sample size)
window_sizes = [3, 4, 5, 6, 7, 8, 9, 10]

# Load your empirical dataset
df = pd.read_csv("2021_2025_disposable_income.csv")

# equivalize income and weight
df["weight"] = df["weight"] * df["size"]
df["income"] = df["income"] / np.sqrt(df["size"])


In [7]:
results = []

for year, group in df.groupby("year"): # Changed 'grp' to 'group'
    y_val = group["income"]
    w_val = group["weight"]

    # 1. Calculate the total weight N
    N = np.sum(w_val)
    target_weight = p * N

    # 2. Aggregate into unique support points and sort
    support_df = pd.DataFrame({'y': y_val, 'w': w_val}).groupby('y').sum().sort_index()

    # 3. Calculate cumulative sum of the unique weights
    cum_w = support_df['w'].cumsum().values

    # 4. Find the maximum k
    k_max = np.searchsorted(cum_w, target_weight, side='right')

    # Ensure k_range is valid
    k_max = max(k_min + 1, k_max)
    k_range_obj = range(k_min, k_max)

    # Run the analysis in parallel
    # MATCHED ARGUMENT ORDER: compute_single_scenario(income, weight, k_range, window_size)
    rs = Parallel(n_jobs=n_cores)(
        delayed(compute_single_scenario)(y_val, w_val, k_range_obj, ws) for ws in window_sizes
    )

    # Append the year to each dictionary in the returned list
    for res_dict in rs:
        res_dict['year'] = year
        results.append(res_dict)

print(pd.DataFrame(results))

     w     gamma_hat   k^*  year
0    3 -38180.703019   988  2021
1    4 -38180.703019   988  2021
2    5 -38180.703019   988  2021
3    6 -38180.703019   988  2021
4    7 -38180.703019   988  2021
5    8 -38180.703019   988  2021
6    9 -38180.703019   988  2021
7   10 -38180.703019   988  2021
8    3  -5853.680013  1032  2022
9    4  -5853.680013  1032  2022
10   5  -5853.680013  1032  2022
11   6  -5853.680013  1032  2022
12   7  -5853.680013  1032  2022
13   8  -5853.680013  1032  2022
14   9  -5853.680013  1032  2022
15  10  -5853.680013  1032  2022
16   3 -21090.098056  1039  2023
17   4 -21090.098056  1039  2023
18   5 -21090.098056  1039  2023
19   6 -21090.098056  1039  2023
20   7 -21090.098056  1039  2023
21   8 -21090.098056  1039  2023
22   9 -21090.098056  1039  2023
23  10 -21090.098056  1039  2023
24   3 -18489.749215  1086  2024
25   4 -18489.749215  1086  2024
26   5 -18489.749215  1086  2024
27   6 -18489.749215  1086  2024
28   7 -18489.749215  1086  2024
29   8 -18